# Compositional binding vs. text-only shortcuts in CLIP — full pipeline

YZM5110 research project. This notebook runs every experiment of the repository
[moized/clip-compositional-hardnegs](https://github.com/moized/clip-compositional-hardnegs):

1. reproduction of ARO / SugarCrepe results with the released checkpoints,
2. text-only and shuffled-image controls,
3. construction of the training negatives,
4. the fine-tuning grid (5 conditions × 3 seeds) with evaluation,
5. all tables and figures.

## Before you run

**Notebook settings (right-hand panel)**
- **Accelerator:** GPU T4 ×2 (one T4 also works).
- **Internet:** On (your Kaggle account must be phone-verified).

**Optional inputs (Add Input)** — nothing is required, everything needed is downloaded.
- Any COCO dataset with `train2014` / `val2014` / `train2017` / `val2017` folders: images found there are not downloaded again.
- The competition data *ImageNet Object Localization Challenge*: adds ImageNet to the zero-shot evaluation.

**How to use it**
1. First leave `QUICK = True` in the next cell and choose *Run All*. This is a rehearsal of every step on a small sample (about 15 minutes); its numbers mean nothing and go to `results_quick/`.
2. Finish `HYPOTHESES.md` in the repository, set its status to `FINAL`, commit and push. The fine-tuning grid refuses to start before that.
3. Set `QUICK = False` and use *Save Version → Save & Run All (Commit)*. The full run takes about 6 hours on a T4 and continues when the browser is closed.
4. Download `results.zip` from the notebook's *Output* tab and commit its contents to the repository (last cell).

In [ ]:
# ---- Settings you may change ------------------------------------------------------------
REPO_URL    = "https://github.com/moized/clip-compositional-hardnegs.git"
REPO_BRANCH = "claude/research-code"   # write "main" here once the branch has been merged
QUICK       = True     # True: rehearsal on a small sample.  False: the real experiment.
RUN_GRID    = True     # False: stop after the reproduction, the controls and the training data
IMAGENET_LIMIT = None  # e.g. 10000 to use a fixed random subset of the ImageNet validation set
# ------------------------------------------------------------------------------------------
import glob, json, os, shutil, subprocess, sys

ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK      = "/kaggle/working" if ON_KAGGLE else os.getcwd()
SCRATCH   = "/kaggle/temp" if ON_KAGGLE else os.path.join(WORK, "scratch")
REPO_DIR  = os.path.join(WORK, "clip-compositional-hardnegs")
DATA      = os.path.join(SCRATCH, "data")     # downloads: large, not saved with the notebook
RUNS      = os.path.join(SCRATCH, "runs")     # checkpoints: large, deleted after evaluation
RESULTS   = os.path.join(REPO_DIR, "results_quick" if QUICK else "results")
CONFIG    = "configs/quick.yaml" if QUICK else "configs/main.yaml"
COCO_ROOTS = "/kaggle/input"                  # attached datasets; missing images are downloaded
IMAGENET  = "/kaggle/input/imagenet-object-localization-challenge"

os.makedirs(DATA, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
os.environ["TQDM_MININTERVAL"] = "30"         # fewer progress-bar lines in the saved log
os.environ["TOKENIZERS_PARALLELISM"] = "false"
PY = sys.executable


def run(command):
    # Run a shell command, show its output as it arrives, stop the notebook if it fails.
    print("$", command, flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {process.returncode}): {command}")


print("rehearsal on a small sample" if QUICK else "REAL EXPERIMENT", "| results go to", RESULTS)

## 0. Get the code

The repository is cloned, and every step below calls one of its scripts. This is
better than pasting code into cells with `%%writefile`: the code that produces the
numbers is exactly the code in the public repository, each result file records the
commit it was made with, and there is only one copy to keep correct. (If you want to
try a change inside Kaggle, `%%writefile path/to/file.py` still works on the cloned
files, but the change stays in this session until you commit it to GitHub.)

In [ ]:
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    for branch in (REPO_BRANCH, "main"):
        if subprocess.call(["git", "clone", "--depth", "1", "--branch", branch, REPO_URL, REPO_DIR]) == 0:
            break
    else:
        raise RuntimeError("Could not clone the repository. Is Internet switched on in the notebook settings?")
os.chdir(REPO_DIR)
os.makedirs(RESULTS, exist_ok=True)
run("git log -1 --format='code version: %H  (%cd)' --date=short")

In [ ]:
# Packages that Kaggle's image does not have, the spaCy model, and a record of all versions.
run(f"{PY} -m pip install -q -r requirements-kaggle.txt")
try:
    import spacy
    spacy.load("en_core_web_sm")
except Exception:
    run(f"{PY} -m spacy download en_core_web_sm")
run(f"{PY} -m pip freeze > {RESULTS}/environment.txt")

import torch
print("PyTorch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not torch.cuda.is_available() and not QUICK:
    raise RuntimeError("No GPU. Choose Accelerator = GPU T4 x2 in the notebook settings.")

In [ ]:
# Unit tests (seconds) and a quick end-to-end check with the real model.
run(f"{PY} -m pytest -q tests")
run(f"{PY} scripts/quick_test.py --data-dir {DATA} --coco-roots {COCO_ROOTS}")

## 1. Download the data

Benchmarks, the released NegCLIP checkpoint, split files and the COCO images used
for evaluation (about 5 GB). Files that already exist are skipped.

If Google Drive refuses a file ("too many users have viewed or downloaded this file"),
the script prints the link and where the file must go. Download it on your computer,
upload it to Kaggle as a private dataset, add it as an input and copy it to that
place, for example:
`shutil.copy("/kaggle/input/<your-dataset>/negclip.pth", f"{DATA}/ckpt/negclip.pth")`.
Then run the cell again.

In [ ]:
run(f"{PY} scripts/00_download_data.py --data-dir {DATA} --coco-roots {COCO_ROOTS}")
shutil.copy(os.path.join(DATA, "data_manifest.json"), os.path.join(RESULTS, "data_manifest.json"))

## 2. Reproduction and controls

- OpenAI CLIP ViT-B/32 and the released NegCLIP on VG-Relation, VG-Attribution, COCO-Order and SugarCrepe.
  NegCLIP is evaluated twice: with the activation function it was trained with (QuickGELU) and with the one
  the released evaluation code uses (GELU).
- Zero-shot classification and COCO retrieval for both.
- Text-only baselines: a language model picks the caption without seeing the image.

In [ ]:
COMMON = f"--data-dir {DATA} --coco-roots {COCO_ROOTS} --out-dir {RESULTS}"
SAMPLE = "--limit 500 --order-seeds 0" if QUICK else ""       # rehearsal: 500 items per benchmark

run(f"{PY} scripts/01_eval_compositional.py --model openai  --name clip         {COMMON} {SAMPLE}")
run(f"{PY} scripts/01_eval_compositional.py --model negclip --name negclip      {COMMON} {SAMPLE}")
run(f"{PY} scripts/01_eval_compositional.py --model negclip --name negclip_gelu --gelu {COMMON} {SAMPLE}")

In [ ]:
ZERO_SHOT = "cifar10" if QUICK else "cifar10 cifar100"
if os.path.isdir(IMAGENET) and not QUICK:
    ZERO_SHOT += f" imagenet1k --imagenet-root {IMAGENET}"
    if IMAGENET_LIMIT:
        ZERO_SHOT += f" --imagenet-limit {IMAGENET_LIMIT}"
else:
    print("ImageNet is not attached (or this is a rehearsal): zero-shot uses CIFAR only.")

for model, name in (("openai", "clip"), ("negclip", "negclip")):
    run(f"{PY} scripts/05_eval_downstream.py --model {model} --name {name} {COMMON} --zero-shot {ZERO_SHOT}")

In [ ]:
run(f"{PY} scripts/02_eval_text_only.py {COMMON} {SAMPLE}")

In [ ]:
from IPython.display import Image, Markdown, display


def show_tables(*names):
    for name in names:
        path = os.path.join(RESULTS, "tables", f"{name}.md")
        if os.path.isfile(path):
            display(Markdown(open(path, encoding="utf-8").read()))


def show_figures(*names):
    for name in names:
        path = os.path.join(RESULTS, "figures", f"{name}.png")
        if os.path.isfile(path):
            display(Image(path, width=900))


run(f"{PY} scripts/06_make_tables.py --results-dir {RESULTS} > /dev/null")
run(f"{PY} scripts/07_make_figures.py --results-dir {RESULTS}")
show_tables("reproduction_aro", "reproduction_sugarcrepe", "reproduction_text_only", "controls",
            "per_relation", "slices_text_only", "slices_seen_images")
show_figures("fig1_reproduction", "fig2_controls")

## 3. Training data

Captions of COCO train2014 images that no benchmark uses; all NegCLIP-style swaps for
each caption; a fluency gap from GPT-2; one fluency-matched pair per image. The script
stops if a training image appears in any test set.

After this cell, label the audit sample (`results/training_data/audit_sample.csv`,
see `docs/AUDIT.md`). It takes about half an hour and can be done while the grid runs.

In [ ]:
CHECK = "--check-against-negclip" + (" --check-sample 2000" if QUICK else "")
run(f"{PY} scripts/03_build_training_data.py --config {CONFIG} --data-dir {DATA} --coco-roots {COCO_ROOTS} "
    f"--results-dir {RESULTS} --spacy-procs 2 {CHECK}")
run(f"{PY} scripts/06_make_tables.py --results-dir {RESULTS} > /dev/null")
run(f"{PY} scripts/07_make_figures.py --results-dir {RESULTS}")
show_tables("training_negatives")
show_figures("fig3_fluency_gap")

import pandas as pd
pairs = pd.read_csv(os.path.join(DATA, "training", "pairs.csv.gz"))
print(pairs["split"].value_counts().to_dict())
with pd.option_context("display.max_colwidth", 90):
    display(pairs[["caption", "neg_fluent", "gap_fluent", "neg_swap"]].sample(8, random_state=0))

## 4. Fine-tuning grid

Every condition is trained from the same OpenAI weights on the same images and
captions for the same number of steps; only the negative caption differs. After each
run the model is evaluated on the four benchmarks and on the downstream tasks, and
its checkpoint is deleted. Finished steps are skipped, so the cell can be run again
after an interruption (within the same session).

The real grid starts only when `HYPOTHESES.md` in the repository says `STATUS: FINAL`.

In [ ]:
hypotheses = open("HYPOTHESES.md", encoding="utf-8").read()
if not QUICK and "STATUS: FINAL" not in hypotheses:
    raise RuntimeError("HYPOTHESES.md is still a draft. Finish it, set 'STATUS: FINAL', commit and push it, "
                       "then start a new session so that the notebook clones the committed version.")

if RUN_GRID:
    extra = ""
    if os.path.isdir(IMAGENET) and not QUICK:
        extra = f" --zero-shot cifar10 cifar100 imagenet1k --imagenet-root {IMAGENET}"
        if IMAGENET_LIMIT:
            extra += f" --imagenet-limit {IMAGENET_LIMIT}"
    run(f"{PY} scripts/run_grid.py --config {CONFIG} --data-dir {DATA} --coco-roots {COCO_ROOTS} "
        f"--runs-dir {RUNS} --results-dir {RESULTS}{extra}")

## 5. Tables and figures

In [ ]:
run(f"{PY} scripts/06_make_tables.py --results-dir {RESULTS} > /dev/null")
run(f"{PY} scripts/07_make_figures.py --results-dir {RESULTS}")
print(open(os.path.join(RESULTS, "tables", "README.md"), encoding="utf-8").read())
show_tables("conditions_compositional", "conditions_differences", "conditions_slices", "cost", "validation")
show_figures("fig4_conditions", "fig5_text_slices", "fig6_validation")

## 6. Save the results

`results.zip` appears in the notebook's *Output* tab (after *Save & Run All*, open the
saved version to find it). On your computer:

1. unzip it into your clone of the repository, so that the files land in `results/`;
2. `git add results`, commit and push;
3. fill in the audit labels, run `python scripts/08_audit_summary.py`, commit again;
4. when the report is finished, tag the version it describes: `git tag v1.0` and `git push --tags`.

Results of a rehearsal (`results_quick/`) are not for the report; do not commit them.

In [ ]:
archive = shutil.make_archive(os.path.join(WORK, os.path.basename(RESULTS)), "zip", REPO_DIR, os.path.basename(RESULTS))
size = os.path.getsize(archive) / 1e6
print(f"{archive}  ({size:.1f} MB)")
print("folders:", sorted(os.listdir(RESULTS)))